# ETF Portfolio Backtester - Demo Notebook

**DADS 4002 Course Project**

This notebook demonstrates how to use the ETF Portfolio Backtester system interactively.

---

## 📦 Step 1: Import Libraries and Modules

In [1]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Import project modules
from modules.db_connector import DatabaseConnector
from modules.backtest_engine import MomentumBacktester as BacktestEngine
from modules.analytics import PortfolioAnalytics as Analytics
from modules.crud_operations import CRUDOperations

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ All modules imported successfully!")

✓ All modules imported successfully!


## 🔌 Step 2: Connect to Database

In [2]:
# Cell 2: Connect to Database
from modules.db_connector import DatabaseConfig

# สร้าง config พร้อมข้อมูลที่ถูกต้อง
config = DatabaseConfig()
config.host = '127.0.0.1'           # Hostname
config.port = 3306                   # Port
config.user = 'root'                 # Username
config.password = 'krittanut123456'  # Password ของคุณ
config.database = 'etf_backtester_db'

# เชื่อมต่อด้วย config ที่แก้แล้ว
db = DatabaseConnector(config)

# ทดสอบการเชื่อมต่อ
if db.test_connection():
    print("✓ Database connected successfully!\n")
    
    # ดูจำนวนข้อมูล
    etf_count = db.get_table_count('ETF_Master')
    price_count = db.get_table_count('Price_Data')
    
    print(f"✓ ETF_Master: {etf_count} ETFs")
    print(f"✓ Price_Data: {price_count:,} records")
else:
    print("✗ Cannot connect to database")
    print("Please check MySQL connection settings")

✓ Connection pool 'etf_pool' created successfully
✓ Successfully connected to database 'etf_backtester_db'
✓ Database connected successfully!

✓ ETF_Master: 50 ETFs
✓ Price_Data: 25,933 records


## 📊 Step 3: View ETF Master Data

In [ ]:
# Query all ETFs
query = """
SELECT
    ETF_ID,
    Ticker_Symbol,
    ETF_Name,
    Asset_Type,
    Expense_Ratio
FROM ETF_Master
ORDER BY Asset_Type, Ticker_Symbol
"""

results = db.execute_query_dict(query)
df_etfs = pd.DataFrame(results)

print(f"Total ETFs: {len(df_etfs)}\n")

# Display first 10
print("Sample ETFs:")
print(df_etfs.head(10))

# Count by Asset Type
print("\nETF Count by Asset Type:")
print(df_etfs.groupby('Asset_Type').size())

## 📈 Step 4: View Recent Price Data (SPY Example)

In [ ]:
# Query SPY recent prices
query = """
SELECT
    pd.Price_Date,
    pd.Open_Price,
    pd.High_Price,
    pd.Low_Price,
    pd.Close_Price,
    pd.Volume,
    em.Ticker_Symbol
FROM Price_Data pd
JOIN ETF_Master em ON pd.ETF_ID = em.ETF_ID
WHERE em.Ticker_Symbol = 'SPY'
ORDER BY pd.Price_Date DESC
LIMIT 10
"""

results = db.execute_query_dict(query)
df_spy = pd.DataFrame(results)

print("SPY - Latest 10 Weeks:")
print(df_spy)

# Plot
df_spy_sorted = df_spy.sort_values('Price_Date')
plt.figure(figsize=(12, 6))
plt.plot(df_spy_sorted['Price_Date'], df_spy_sorted['Close_Price'], marker='o', linewidth=2)
plt.title('SPY - Close Price (Last 10 Weeks)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 🎯 Step 5: Run Backtest

In [4]:
# Cell 5: Run Backtest
# Initialize backtest engine
backtest = BacktestEngine(db)

# Set parameters
# ใช้ช่วงเวลา 6 เดือนล่าสุด
from datetime import datetime, timedelta

end_date = datetime.now()
start_date = end_date - timedelta(days=180)  # ย้อนหลัง 6 เดือน

lookback_days = 90
top_n = 5
holding_period_days = 30
rebalance_days = 30

print("Running Backtest...")
print("=" * 60)
print(f"Start Date: {start_date.strftime('%Y-%m-%d')}")
print(f"End Date: {end_date.strftime('%Y-%m-%d')}")
print(f"Lookback Period: {lookback_days} days")
print(f"Top N ETFs: {top_n}")
print(f"Holding Period: {holding_period_days} days")
print(f"Rebalance Every: {rebalance_days} days")
print("=" * 60)

# Run backtest
results = backtest.run_backtest(
    start_date=start_date.strftime('%Y-%m-%d'),
    end_date=end_date.strftime('%Y-%m-%d'),
    lookback_days=lookback_days,
    holding_period_days=holding_period_days,
    rebalance_days=rebalance_days,
    top_n=top_n
)

# Display results
if results and 'total_return' in results:
    print("\n✓ Backtest completed successfully!")
    
    # Summary
    print(f"\n{'='*60}")
    print("BACKTEST SUMMARY")
    print(f"{'='*60}")
    print(f"Total Return: {results['total_return']:.2f}%")
    print(f"Number of Rebalances: {results.get('num_rebalances', 'N/A')}")
    
    # Show portfolio selections (if available)
    if 'portfolios' in results and len(results['portfolios']) > 0:
        print(f"\nLast Portfolio Selection:")
        last_portfolio = results['portfolios'][-1]
        
        if 'etfs' in last_portfolio:
            df_last = pd.DataFrame(last_portfolio['etfs'])
            print(df_last[['ticker', 'etf_name', 'asset_type', 'momentum_score']])
    
    print(f"\n{'='*60}")
else:
    print("✗ Backtest failed or returned no results")
    print("This might be because there's not enough data in the date range")

Running Backtest...
Start Date: 2025-05-24
End Date: 2025-11-20
Lookback Period: 90 days
Top N ETFs: 5
Holding Period: 30 days
Rebalance Every: 30 days

Running Momentum Strategy Backtest
Backtest Run ID: RUN_20251120_202455_85abdd5f
Date Range: 2025-05-24 to 2025-11-20
Lookback Period: 90 days
Holding Period: 30 days
Rebalance Frequency: 30 days
Portfolio Size: Top 5 ETFs

------------------------------------------------------------

Rebalance Date: 2025-05-24
  ✓ Selected 5 ETFs
    1. GLD    (Commodity ) | Momentum:  17.65% | Price: $309.75
    2. ARKW   (Equity    ) | Momentum:  11.10% | Price: $118.94
    3. EFA    (Equity    ) | Momentum:   7.92% | Price: $88.04
    4. VEA    (Equity    ) | Momentum:   7.75% | Price: $55.03
    5. SLV    (Commodity ) | Momentum:   7.56% | Price: $30.45
  ✓ Logged 5 selections to database
  ✓ Portfolio Return: 6.32%

Rebalance Date: 2025-06-23
  ✓ Selected 5 ETFs
    1. ARKW   (Equity    ) | Momentum:  51.15% | Price: $145.65
    2. ARKF   (Equity

## 📊 Step 6: Analytics - Volatility Analysis

In [8]:
# Cell 6: Analytics - Volatility Analysis

from datetime import datetime, timedelta

analytics = Analytics(db)

print("\n" + "="*60)
print("ANALYTICS: Volatility Analysis by Asset Type")
print("="*60)

# Analyze volatility for all data
volatility_df = analytics.analyze_volatility_by_asset_type()

# Convert to DataFrame for easier viewing
if volatility_df:
    import pandas as pd
    df = pd.DataFrame(volatility_df)
    print("\n✓ Volatility analysis complete!")
else:
    print("\n✗ No volatility data found")


ANALYTICS: Volatility Analysis by Asset Type

INSIGHT #1: VOLATILITY ANALYSIS BY ASSET TYPE
Analysis Period: Full historical data

--------------------------------------------------------------------------------
Asset Type      # ETFs   Obs      Avg Weekly % Weekly Vol % Annual Vol %   
--------------------------------------------------------------------------------
Commodity       5        2605          0.1690      3.3861          24.42
Equity          25       12858         0.2482      3.2978          23.78
Mixed           5        2605          0.0650      1.3833           9.97
Bond            15       7815         -0.0085      1.0568           7.62
--------------------------------------------------------------------------------

✓ INSIGHT: 'Commodity' has the HIGHEST volatility at 24.42% (annualized)
✓ INSIGHT: 'Bond' has the LOWEST volatility at 7.62% (annualized)

✓ Volatility analysis complete!


## 📊 Step 7: Analytics - Lookback Period Optimization

In [9]:
# Cell 7: Analytics - Lookback Period Optimization

print("\n" + "="*60)
print("ANALYTICS: Lookback Period Optimization")
print("="*60)

# Compare different lookback periods (90 days vs 180 days)
lookback_comparison = analytics.compare_lookback_periods([90, 180])

if lookback_comparison:
    df_lookback = pd.DataFrame(lookback_comparison)
    print("\n✓ Lookback period analysis complete!")
else:
    print("\n✗ No lookback comparison data available")


ANALYTICS: Lookback Period Optimization

INSIGHT #2: LOOKBACK PERIOD OPTIMIZATION (CAGR COMPARISON)

--------------------------------------------------------------------------------
Lookback     # Runs   Avg Rebal    Cum Return %    CAGR %     Days    
--------------------------------------------------------------------------------
90 days      8                4.0          40.76    183.07 120     
--------------------------------------------------------------------------------

✓ INSIGHT: Lookback period of 90 days yielded the HIGHEST CAGR at 183.07%

✓ Lookback period analysis complete!


## 📊 Step 8: Analytics - Drawdown Analysis

In [7]:
# Cell 8: Analytics - Drawdown Analysis

analytics = Analytics(db)  # Create analytics object

print("\n" + "="*60)
print("ANALYTICS: Drawdown Exposure Analysis")
print("="*60)

# Analyze which asset types were held during maximum drawdowns
drawdown_analysis = analytics.analyze_drawdown_exposure()

if drawdown_analysis:
    df_drawdown = pd.DataFrame(drawdown_analysis)
    print("\n✓ Drawdown analysis complete!")
else:
    print("\n✗ No drawdown data available")


ANALYTICS: Drawdown Exposure Analysis

INSIGHT #3: ASSET TYPE EXPOSURE DURING MAXIMUM DRAWDOWNS
Analysis Scope: All backtest runs

--------------------------------------------------------------------------------
Asset Type      # Times    Avg Holdings  Avg Weight %  Contribution % 
--------------------------------------------------------------------------------
Commodity       8                   1.5        20.00         0.8745
Equity          8                   4.2        20.00         4.8758
--------------------------------------------------------------------------------

✓ INSIGHT: 'Commodity' was most frequently held during maximum drawdown periods (8 times)
✓ INSIGHT: 'Commodity' had the worst contribution during drawdowns (0.8745%)

✓ Drawdown analysis complete!


## 🔍 Step 9: CRUD Operations - View Latest Backtest

In [ ]:
# Cell 9: CRUD Operations - View Latest Backtest
crud = CRUDOperations(db)

print("\n" + "="*60)
print("CRUD: View Latest Backtest Results")
print("="*60)

# Read latest backtest results (uses correct method name)
results = crud.read_backtest_results(limit=50)

if results:
    # Convert to DataFrame for analysis
    df_results = pd.DataFrame(results)
    
    # Show summary statistics
    print("\n📊 Summary Statistics:")
    print(f"  Total Selections: {len(df_results)}")
    print(f"  Unique Dates: {df_results['Selection_Date'].nunique()}")
    print(f"  Average Momentum Score: {df_results['Momentum_Score'].mean():.2f}%")
    
    if df_results['Holding_Return'].notna().any():
        avg_return = df_results['Holding_Return'].mean() * 100
        print(f"  Average Holding Return: {avg_return:.2f}%")
    
    print("\n✓ Latest backtest displayed successfully!")
else:
    print("\n✗ No backtest results found")

## 📈 Step 10: Advanced Analysis - Cumulative Returns

In [ ]:
# Get 1 year of data for top 5 ETFs
query = """
SELECT
    em.Ticker_Symbol,
    em.Asset_Type,
    pd.Price_Date,
    pd.Close_Price
FROM Price_Data pd
JOIN ETF_Master em ON pd.ETF_ID = em.ETF_ID
WHERE pd.Price_Date >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR)
  AND em.Ticker_Symbol IN ('SPY', 'QQQ', 'AGG', 'GLD', 'TLT')
ORDER BY em.Ticker_Symbol, pd.Price_Date
"""

results = db.execute_query_dict(query)
df = pd.DataFrame(results)

if not df.empty:
    # Pivot table
    pivot = df.pivot_table(
        index='Price_Date',
        columns='Ticker_Symbol',
        values='Close_Price'
    )
    
    # Calculate returns
    returns = pivot.pct_change()
    
    # Calculate cumulative returns
    cumulative_returns = (1 + returns).cumprod()
    
    # Plot
    plt.figure(figsize=(14, 7))
    
    for ticker in cumulative_returns.columns:
        plt.plot(cumulative_returns.index, cumulative_returns[ticker], 
                label=ticker, linewidth=2)
    
    plt.title('Cumulative Returns - Last 1 Year', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\nSummary Statistics (Annualized):")
    annual_returns = returns.mean() * 52
    annual_vol = returns.std() * (52 ** 0.5)
    
    summary = pd.DataFrame({
        'Annual Return': annual_returns,
        'Annual Volatility': annual_vol,
        'Sharpe Ratio': annual_returns / annual_vol
    })
    
    print(summary)
else:
    print("No data available")

## 🔒 Step 11: Close Database Connection

In [ ]:
# Close database connection
db.close_pool()
print("✓ Database connection closed successfully")
print("\nThank you for using ETF Portfolio Backtester!")

---

## 📚 Notes

- **All data is REAL from Yahoo Finance** (not synthetic)
- **50 real ETFs** across 4 asset types
- **10 years** of weekly price data
- **~25,933 price records** in total

---

**DADS 4002 Course Project** | Made with ❤️